In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Оценка стоимости кварталов

In [ ]:
blocks_gdf = gdf = pd.read_pickle("../data/traning_data/blocks.pickle")
blocks_gdf.columns

In [ ]:
cadastr_blocks = gpd.read_parquet('../data/traning_data/spb_bloks_price.parquet')
cadastr_blocks.head()

In [ ]:
# Список всех категорий и количество значений в каждой (включая NaN)
col = "permitted_use_established_by_document"

counts = (
    cadastr_blocks[col]
    .value_counts(dropna=False)
    .rename_axis(col)
    .reset_index(name="count")
)
print(counts.to_string(index=False, justify="right"))



In [ ]:
cadastr_blocks['log_cost_index'] = np.log1p(cadastr_blocks['cost_index'])

In [ ]:
# Картируем лог-цену
cadastr_blocks.plot(
    column='log_cost_index',
    scheme='quantiles',
    legend=True,
    figsize=(15,15),
    cmap='viridis'
).set_axis_off()
plt.title('Карта кадастровой стоимости земельных участков в Санкт-Петербурге (логарифм от стоимости м2)')

plt.show()

In [ ]:
cadastr_blocks

In [ ]:
import geopandas as gpd

# — 1. Клонируем оригиналы, чтобы не портить исходники
blocks = blocks_gdf.copy()
cad    = cadastr_blocks.copy()

# — 2. Приводим к единому CRS (он нужен для корректных площадей)
if blocks.crs != cad.crs:
    cad = cad.to_crs(blocks.crs)

# — 3. Считаем полную площадь cadastral-полигонов
cad['cad_area'] = cad.geometry.area

# — 4. Явно добавляем в blocks колонку block_id из индекса
blocks['block_id'] = blocks.index

# — 5. Пересечение кварталов и кадастра
inter = gpd.overlay(
    blocks[['block_id', 'geometry']],
    cad   [['cost_value', 'cad_area', 'geometry']],
    how='intersection'
)

# — 6. Площадь каждого куска пересечения
inter['int_area'] = inter.geometry.area

# — 7. Вклады стоимости пропорционально площади
inter['cost_contrib'] = inter['cost_value'] * inter['int_area'] / inter['cad_area']

# — 8. Суммируем для каждого квартала
block_costs = (
    inter
    .groupby('block_id', as_index=False)['cost_contrib']
    .sum()
    .rename(columns={'cost_contrib':'land_value'})
)

# — 9. Мерджим обратно в blocks, заполняем пропуски нулями
blocks = blocks.merge(block_costs, on='block_id', how='left')
blocks['land_value'] = blocks['land_value'].fillna(0)

# — 10. При желании убираем служебный block_id (или возвращаем исходный индекс)
blocks = blocks.drop(columns=['block_id'])
blocks


In [ ]:
# 1. Если CRS географический (deg), переведём в метрический (например, Меркатор)
if blocks.crs.is_geographic:
    blocks = blocks.to_crs(epsg=3857)

# 2. Считаем площадь каждого квартала в м²
blocks['site_area'] = blocks.geometry.area
blocks = blocks.to_crs(epsg=32636)



In [ ]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# blocks['log_std'] = scaler.fit_transform(
#     blocks[['log_price']]
# )

In [ ]:
# from sklearn.cluster import KMeans

# # Берём только ненулевые и не NaN
# X = blocks[['log_price']].dropna()

# kmeans = KMeans(n_clusters=6, random_state=0)
# blocks.loc[X.index, 'cluster'] = kmeans.fit_predict(X)

# # Визуализируем
# blocks.plot(
#     column='cluster',
#     categorical=True,
#     legend=True,
#     figsize=(15,15)
# ).set_axis_off()
# plt.title('Кластеры кварталов по цене за квадратный метр')
# plt.show()


In [ ]:
# Собираем все имена колонок, которые начинаются на 'capacity'
cols_to_drop = [col for col in blocks.columns if col.startswith('capacity')]
blocks = blocks.drop(columns=cols_to_drop)
cols_to_drop = [col for col in blocks.columns if col.startswith('count')]
# blocks = blocks.drop(columns='cluster')
# Удаляем их сразу все
blocks = blocks.drop(columns=cols_to_drop)

blocks.columns

In [ ]:
blocks['land_use'] = (
    blocks['land_use']
    .astype(str)
    .str.replace(r'^LandUse\.', '', regex=True)
)

## Дополнитльные метрики

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators 

blocks_df = calculate_density_indicators(blocks[[
    'site_area',
    'footprint_area', 
    'build_floor_area',
    'living_area',
    'non_living_area'
]])
blocks_df

In [ ]:
blocks.loc[:, blocks_df.columns] = blocks_df
blocks

In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes


blocks_df = get_strelka_morphotypes(blocks)
blocks_df

In [ ]:
blocks.loc[:, blocks_df.columns] = blocks_df
blocks

In [ ]:
from blocksnet.relations import generate_adjacency_graph

adjacency_graph = generate_adjacency_graph(blocks)

In [ ]:
import pandas as pd

accessibility_matrix = pd.read_pickle('../data/traning_data/acc_mx.pickle')
accessibility_matrix.head()

In [ ]:
from blocksnet.analysis.network.accessibility import area_accessibility

area_acc_df = area_accessibility(accessibility_matrix, blocks)
area_acc_df

In [ ]:
blocks= blocks.join(area_acc_df)


In [ ]:
blocks.plot(column='area_accessibility', legend=True).set_axis_off()

In [ ]:
# 1. Выведем все dtypes
print(blocks.dtypes)

# 2. Отберём «чисто» категориальные (object / category)
cat_cols = blocks.select_dtypes(include=['object','category']).columns.tolist()
print("Категориальные по dtype:", cat_cols)

# 3. Часто к категориальным относят и числовые колонки с малым числом уникальных значений.
#    Например, если уникальных значений меньше 10:
low_cardinality = [
    col for col in blocks.select_dtypes(include=['int64','float64']).columns
    if blocks[col].nunique() < 10
]
print("Возможно категориальные по низкой кардинальности:", low_cardinality)


In [ ]:
# Очистка базовых невалидных значений
blocks = blocks.replace([np.inf, -np.inf], np.nan).copy()

# Ключевые поля в число
for col in ["land_value", "site_area"]:
    blocks[col] = pd.to_numeric(blocks[col], errors="coerce")

# Удаляем все кварталы, где land_value <= 0 или NaN
blocks = blocks[blocks["land_value"] > 1].copy()

# (Рекомендовано) также убрать нулевую/битую площадь, чтобы деление было корректным
blocks = blocks[blocks["site_area"] > 1].copy()

# Расчёт цены за м² и логов
blocks["land_value_per_sqm"] = blocks["land_value"] / blocks["site_area"]
blocks["log_land_value"] = np.log1p(blocks["land_value"])
blocks["log_land_value_per_sqm"] = np.log1p(blocks["land_value_per_sqm"])

# Быстрая статистика
print(blocks['land_value'].describe())


In [ ]:

# Гистограмма
plt.figure()
blocks['log_land_value'].hist(bins=50)
plt.xlabel('log1p(price_per_sqm)')
plt.ylabel('Частота')
plt.title('Распределение log_land_value')
plt.show()

# Box-plot
plt.figure()
plt.boxplot(blocks['log_land_value'].dropna(), vert=False)
plt.xlabel('log1p(price_per_sqm)')
plt.title('Box-plot log_land_value')
plt.show()

# Картируем лог-цену
blocks.plot(
    column='log_land_value',
    scheme='quantiles',
    legend=True,
    figsize=(15,15),
    cmap='viridis'
).set_axis_off()
plt.title('Карта log_land_value по кварталам')

plt.show()

In [ ]:
import pandas as pd

# 1. Отбираем числовые столбцы и фильтруем
num = blocks.select_dtypes(include=['number'])
num = num[num['land_value'] > 1]

# 2. Считаем count + статистики
agg_stats = num.agg(['count', 'min', 'max', 'mean', 'median', 'std']).T

# 3. Переименовываем столбцы
agg_stats.index.name = 'Variable'
agg_stats = agg_stats.rename(columns={
    'count':  'Count',
    'min':    'Min',
    'max':    'Max',
    'mean':   'Mean',
    'median': 'Median',
    'std':    'SD'
})

# 4. Приводим Count к int
agg_stats['Count'] = agg_stats['Count'].astype(int)

# 5. Округляем остальные метрики
for col in ['Min','Max','Mean','Median','SD']:
    agg_stats[col] = agg_stats[col].round(2)

# 6. Настраиваем глобальный формат для float
pd.options.display.float_format = '{:,.2f}'.format

# 7. Показываем результат
agg_stats


In [ ]:
# 1) есть ли реально ровно нули
print((blocks["land_value_per_sqm"] == 0).sum())

# 2) какие самые маленькие значения на самом деле
print(blocks["land_value_per_sqm"].nsmallest(10))

# 3) сколько значений "почти ноль"
eps = 1e-6
print((blocks["land_value_per_sqm"] <= eps).sum())



In [ ]:
cols_to_drop = [col for col in blocks.columns if col.startswith('capacity')]
cols_to_drop = [col for col in blocks.columns if col.startswith('count')]
# blocks = blocks.drop(columns='cluster')
blocks = blocks.drop(columns=cols_to_drop)

blocks.columns

In [ ]:
# blocks.to_parquet('../data/traning_data/prepared_blocks_spb.parquet')

In [ ]:
import libpysal
from esda.moran import Moran, Moran_Local
import matplotlib.pyplot as plt
from splot.esda import lisa_cluster

# 1) Подготовка: spatial weights
# Если у вас уже есть w1 (Rook или Queen), используем его, иначе:
w1 = libpysal.weights.Queen.from_dataframe(blocks_default)
# или: w1 = libpysal.weights.Rook.from_dataframe(blocks)
w1.transform = 'r'   # row-standardization

# 2) Извлекаем вектор переменной
y = blocks_default['log_total_price'].values

# 3) Глобальный индекс Морена
moran_global = Moran(y, w1)
print("Global Moran’s I:",     round(moran_global.I,3))
print("p-value (normal):",     round(moran_global.p_norm,3))
print("z-score (normal):",     round(moran_global.z_norm,3))
print("p-value (permut.):",    round(moran_global.p_sim,3))
print("z-score (permut.):",    round(moran_global.z_sim,3))

# 4) Локальный индекс Морена
lisa = Moran_Local(y, w1)

# 5) Добавляем результаты в GeoDataFrame
blocks_default['lisa_I']   = lisa.Is          # локальные значения I
blocks_default['lisa_p']   = lisa.p_sim       # p-values по перестановкам
blocks_default['lisa_q']   = lisa.q           # квадранты (1=HH, 2=LH, 3=LL, 4=HL)
blocks_default['lisa_sig'] = lisa.p_sim < 0.05

# 6) Визуализация LISA-кластеров
fig, ax = plt.subplots(1, figsize=(15, 15))
# splot умеет сам раскрасить по четырём кластерам
lisa_cluster(lisa, blocks_default, p=0.05, ax=ax)
ax.set_title("LISA-кластеризация (p<0.05)")
ax.axis('off')
plt.show()


In [ ]:
blocks_default